# Cluster AgentForge SWE-bench Run

Use this notebook on an interactive cluster node to preview, collect, evaluate, and inspect SWE-bench Verified trajectories. It uses the repository scripts and pinned dataset configuration rather than reimplementing the pipeline in notebook cells.

Reuse the same `RUN_NAME` to resume compatible output; choose a new name after changing result-affecting settings. Overwrite remains disabled by default.

## 0. Setup

Recommended cluster flow: start vLLM separately, wait for `/v1/models`, run the one-instance smoke below, then run a shard. If you want this notebook to launch the Apptainer vLLM helper, set `START_VLLM=1` before running cell 3.

In [ ]:
from datetime import datetime
from pathlib import Path
import glob
import json
import os
import shlex
import shutil
import subprocess
import sys


def find_repo_root():
    raw_candidates = [
        os.getenv('DEBUG_DEPO_ROOT'),
        Path.cwd(),
        Path.cwd() / 'debug-depo',
        Path.cwd().parent,
        Path.home() / 'debug-depo',
    ]
    candidates = []
    for candidate in raw_candidates:
        if not candidate:
            continue
        path = Path(candidate).expanduser().resolve()
        if path not in candidates:
            candidates.append(path)
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    searched = '\n'.join(f'  - {candidate}' for candidate in candidates)
    raise FileNotFoundError(f'Could not find debug-depo repo root. Searched:\n{searched}')


def find_ephemeral_root(root):
    raw_candidates = [
        os.getenv('DEBUG_DEPO_EPHEMERAL'),
        Path(os.environ['RDS']) / 'ephemeral' / 'debug-depo' if os.getenv('RDS') else None,
        Path(os.environ['EPHEMERAL']) / 'debug-depo' if os.getenv('EPHEMERAL') else None,
        Path(os.environ['SCRATCH']) / 'debug-depo' if os.getenv('SCRATCH') else None,
        root / 'scratch',
    ]
    for candidate in raw_candidates:
        if not candidate:
            continue
        path = Path(candidate).expanduser().resolve()
        if path.exists() or path.parent.exists():
            return path
    return (root / 'scratch').resolve()


ROOT = find_repo_root()
kernel_bin = Path(sys.executable).resolve().parent
kernel_uv = kernel_bin / 'uv'
ephemeral = find_ephemeral_root(ROOT)
scratch = Path(os.getenv('DEBUG_DEPO_SCRATCH') or ephemeral).expanduser().resolve()

env = os.environ.copy()
env['PATH'] = str(kernel_bin) + os.pathsep + env.get('PATH', '')
env['PYTHONPATH'] = str(ROOT / 'src') + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
env.setdefault('DEBUG_DEPO_ROOT', str(ROOT))
env.setdefault('DEBUG_DEPO_EPHEMERAL', str(ephemeral))
env.setdefault('DEBUG_DEPO_SCRATCH', str(scratch))
env.setdefault('HF_HOME', str(scratch / 'huggingface'))
env.setdefault('UV_CACHE_DIR', str(scratch / 'uv-cache'))
env.setdefault('TMPDIR', str(scratch / 'tmp'))
if kernel_uv.exists():
    env.setdefault('UV', str(kernel_uv))
env.setdefault('APPTAINER_CACHEDIR', str(scratch / 'apptainer-cache'))
env.setdefault('VLLM_IMAGE', str(ROOT / 'cluster' / 'apptainer' / 'vllm-openai.sif'))
env.setdefault('SWEBENCH_APPTAINER_CACHE_DIR', str(scratch / 'swebench_epoch_cache' / 'apptainer-cache'))
env.setdefault('SWEBENCH_APPTAINER_SIF_DIR', str(scratch / 'swebench_epoch_cache' / 'sifs'))

for path in [
    scratch,
    Path(env['HF_HOME']),
    Path(env['UV_CACHE_DIR']),
    Path(env['TMPDIR']),
    Path(env['APPTAINER_CACHEDIR']),
    Path(env['VLLM_IMAGE']).parent,
    Path(env['SWEBENCH_APPTAINER_CACHE_DIR']),
    Path(env['SWEBENCH_APPTAINER_SIF_DIR']),
]:
    path.mkdir(parents=True, exist_ok=True)

print(f'Repo: {ROOT}')
print(f'Ephemeral root: {ephemeral}')
print(f'Scratch: {scratch}')
print(f"HF_HOME: {env['HF_HOME']}")
print(f"APPTAINER_CACHEDIR: {env['APPTAINER_CACHEDIR']}")
print(f"TMPDIR: {env['TMPDIR']}")
print(f"UV: {env.get('UV', '<from PATH>')}")
print(f"VLLM_IMAGE: {env['VLLM_IMAGE']}")
print(f'Kernel bin prepended to subprocess PATH: {kernel_bin}')


## 0.1 Python Environment Check

Run this once near the top. The active kernel should be the same Python environment where `uv sync` and `scripts/install_mini_swe_agent_plus.sh` were run. This can be either a Miniforge/Conda environment or an Imperial-style `tools/prod` + `Python` virtualenv.


In [ ]:
conda_prefix = os.environ.get('CONDA_PREFIX')
conda_env = os.environ.get('CONDA_DEFAULT_ENV')
virtual_env = os.environ.get('VIRTUAL_ENV')
subprocess_python = shutil.which('python', path=env['PATH'])
subprocess_uv = env.get('UV') or shutil.which('uv', path=env['PATH'])

print(f'Kernel Python executable: {sys.executable}')
print(f'Kernel bin: {kernel_bin}')
print(f'CONDA_DEFAULT_ENV: {conda_env}')
print(f'CONDA_PREFIX: {conda_prefix}')
print(f'VIRTUAL_ENV: {virtual_env}')
print(f'Python used by notebook subprocess env: {subprocess_python}')
print(f'uv used by notebook subprocess env: {subprocess_uv or "<missing>"}')

if virtual_env == '/opt/jupyter':
    print('Note: VIRTUAL_ENV=/opt/jupyter is the JupyterHub server env, not your selected kernel env.')
if not str(sys.executable).startswith('/opt/jupyter/') and subprocess_python and not subprocess_python.startswith('/opt/jupyter/'):
    print('OK: notebook subprocesses are configured to prefer the selected kernel environment.')
if subprocess_uv is None:
    print('uv is missing from the selected kernel env. That is fine for basic JupyterHub setup, but collect_rollouts.sh needs uv later.')
    print(f'Install it later with: {sys.executable} -m pip install uv')


## 1. Configuration

The defaults match the tracked 500-task Verified PBS workflow. For the official AgentForge 8B model, keep the full Hugging Face id in `AGENTFORGE_MODEL`; the mini-swe LiteLLM model string should be `hosted_vllm/<that full id>`.

In [ ]:
DATASET = os.getenv('DATASET', 'princeton-nlp/SWE-bench_Verified')
DATASET_REVISION = os.getenv('SWEBENCH_DATASET_REVISION', 'c104f840cc67f8b6eec6f759ebc8b2693d585d4a')
SPLIT = os.getenv('SPLIT', 'test')
TASK_IDS_FILE = os.getenv('TASK_IDS_FILE', '')

AGENTFORGE_MODEL = os.getenv('AGENTFORGE_MODEL', 'Kwai-Klear/Klear-AgentForge-8B-SFT')
MINI_SWE_MODEL = os.getenv('MINI_SWE_MODEL', f'hosted_vllm/{AGENTFORGE_MODEL}')
MINI_SWE_CONFIG = os.getenv('MINI_SWE_CONFIG', '')
MINI_SWE_RUNNER = os.getenv('MINI_SWE_RUNNER', 'singularity')
MINI_SWE_ENVIRONMENT_CLASS = os.getenv('MINI_SWE_ENVIRONMENT_CLASS', 'singularity' if MINI_SWE_RUNNER == 'singularity' else '')
MSWEA_SINGULARITY_EXECUTABLE = os.getenv('MSWEA_SINGULARITY_EXECUTABLE', 'apptainer')
LLM_BASE_URL = os.getenv('LLM_BASE_URL', 'http://127.0.0.1:8000/v1').rstrip('/')
LLM_API_KEY = os.getenv('LLM_API_KEY', 'local')

VERIFIED_MODE = os.getenv('VERIFIED_MODE', 'full')
RUN_NAME = os.getenv('RUN_NAME', f'agentforge-verified-{VERIFIED_MODE}-{datetime.now():%Y%m%d}')
RUN_ROOT = Path(os.getenv('RUN_ROOT', scratch / 'runs' / RUN_NAME)).expanduser().resolve()
CLUSTER_LOG_DIR = Path(os.getenv('CLUSTER_LOG_DIR', RUN_ROOT / 'cluster-logs')).expanduser().resolve()
SMOKE_OUTPUT_DIR = Path(os.getenv('SMOKE_OUTPUT_DIR', RUN_ROOT / 'smoke' / 'rollouts'))
SHARD_ROOT = Path(os.getenv('SHARD_ROOT', RUN_ROOT / 'rollouts'))
MERGED_DIR = Path(os.getenv('MERGED_DIR', RUN_ROOT / 'merged'))

SMOKE_LIMIT = int(os.getenv('SMOKE_LIMIT', '5'))
EXPECTED_COUNT = int(os.getenv('EXPECTED_COUNT', '500'))
SMOKE_MAX_STEPS = int(os.getenv('SMOKE_MAX_STEPS', '200'))
FULL_MAX_STEPS = int(os.getenv('MAX_STEPS', '200'))
CONTEXT_LENGTH = int(os.getenv('CONTEXT_LENGTH', '65536'))
TEMPERATURE = float(os.getenv('TEMPERATURE', '0.0'))
TOP_P = float(os.getenv('TOP_P', '1.0'))
TIMEOUT_SECONDS = int(os.getenv('TIMEOUT_SECONDS', '21600'))
MINI_SWE_WORKERS = int(os.getenv('MINI_SWE_WORKERS', '1'))
MINI_SWE_DOCKER_START_CONCURRENCY = int(os.getenv('MINI_SWE_DOCKER_START_CONCURRENCY', '1'))
ROLLOUT_WORKERS = int(os.getenv('ROLLOUT_WORKERS', '8'))

NUM_SHARDS = int(os.getenv('NUM_SHARDS', '10'))
SHARD_INDEX = int(os.getenv('SHARD_INDEX', os.getenv('PBS_ARRAY_INDEX', '0')))

config_preview = {
    'VERIFIED_MODE': VERIFIED_MODE,
    'DATASET': DATASET,
    'DATASET_REVISION': DATASET_REVISION,
    'SPLIT': SPLIT,
    'TASK_IDS_FILE': TASK_IDS_FILE or '<all Verified tasks>',
    'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
    'MINI_SWE_MODEL': MINI_SWE_MODEL,
    'MINI_SWE_RUNNER': MINI_SWE_RUNNER,
    'MINI_SWE_ENVIRONMENT_CLASS': MINI_SWE_ENVIRONMENT_CLASS,
    'MSWEA_SINGULARITY_EXECUTABLE': MSWEA_SINGULARITY_EXECUTABLE,
    'LLM_BASE_URL': LLM_BASE_URL,
    'RUN_NAME': RUN_NAME,
    'RUN_ROOT': str(RUN_ROOT),
    'CLUSTER_LOG_DIR': str(CLUSTER_LOG_DIR),
    'SMOKE_OUTPUT_DIR': str(SMOKE_OUTPUT_DIR),
    'SHARD_ROOT': str(SHARD_ROOT),
    'MERGED_DIR': str(MERGED_DIR),
    'HF_HOME': env['HF_HOME'],
    'APPTAINER_CACHEDIR': env['APPTAINER_CACHEDIR'],
    'TMPDIR': env['TMPDIR'],
    'VLLM_IMAGE': env['VLLM_IMAGE'],
    'SMOKE_MAX_STEPS': SMOKE_MAX_STEPS,
    'FULL_MAX_STEPS': FULL_MAX_STEPS,
    'CONTEXT_LENGTH': CONTEXT_LENGTH,
    'TEMPERATURE': TEMPERATURE,
    'TOP_P': TOP_P,
    'TIMEOUT_SECONDS': TIMEOUT_SECONDS,
    'MINI_SWE_WORKERS': MINI_SWE_WORKERS,
    'ROLLOUT_WORKERS': ROLLOUT_WORKERS,
    'NUM_SHARDS': NUM_SHARDS,
    'SHARD_INDEX': SHARD_INDEX,
}
print(json.dumps(config_preview, indent=2))

if VERIFIED_MODE not in {'smoke', 'full'}:
    raise ValueError(f'VERIFIED_MODE must be smoke or full, got: {VERIFIED_MODE}')
if RUN_ROOT.exists() and any(RUN_ROOT.iterdir()):
    print(f'NOTE: {RUN_ROOT} is not empty. A compatible submission resumes it; otherwise choose a new RUN_NAME.')
if NUM_SHARDS < 1 or NUM_SHARDS > EXPECTED_COUNT:
    raise ValueError('NUM_SHARDS must be between 1 and EXPECTED_COUNT.')
if not 0 <= SHARD_INDEX < NUM_SHARDS:
    raise ValueError(f'Invalid shard {SHARD_INDEX} for NUM_SHARDS={NUM_SHARDS}')


## 2. Optional vLLM Launch

Leave this off if vLLM is already running in another terminal/job. When enabled, this starts `cluster/apptainer/serve_vllm.sh` in the background and writes logs under the smoke output directory.

In [ ]:
START_VLLM = os.getenv('START_VLLM', '0') == '1'
vllm_process = None

if START_VLLM:
    SMOKE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    server_env = env.copy()
    server_env.update({
        'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
        'MINI_SWE_MODEL': MINI_SWE_MODEL,
        'LLM_BASE_URL': LLM_BASE_URL,
        'LLM_API_KEY': LLM_API_KEY,
        'VLLM_MODEL': AGENTFORGE_MODEL,
        'CONTEXT_LENGTH': os.getenv('CONTEXT_LENGTH', '65536'),
    })
    log_path = SMOKE_OUTPUT_DIR / 'vllm.log'
    log_handle = log_path.open('w', encoding='utf-8')
    vllm_process = subprocess.Popen(
        ['bash', 'cluster/apptainer/serve_vllm.sh'],
        cwd=ROOT,
        env=server_env,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        text=True,
    )
    (SMOKE_OUTPUT_DIR / 'vllm.pid').write_text(str(vllm_process.pid))
    print(f'Started vLLM helper pid={vllm_process.pid}; log={log_path}')
else:
    print('START_VLLM is off; expecting an existing server.')

## 3. Check Model Server

Enable `RUN_MODEL_CHECK` before a real rollout. It sends a tiny chat completion to the exact served model id; leave it disabled when only previewing PBS submissions.

In [ ]:
RUN_MODEL_CHECK = False
if RUN_MODEL_CHECK:
    check_cmd = [
        sys.executable,
        '-m',
        'debug_depo.check_local_llm',
        '--base-url', LLM_BASE_URL,
        '--api-key', LLM_API_KEY,
        '--model', AGENTFORGE_MODEL,
        '--timeout', os.getenv('LLM_CHECK_TIMEOUT', '900'),
        '--max-tokens', '16',
    ]
    print(shlex.join(check_cmd))
    subprocess.run(check_cmd, cwd=ROOT, env=env, check=True)
else:
    print('Set RUN_MODEL_CHECK = True before a real rollout.')

## 3.1 Preview or Submit the Tracked PBS Chain

The dry preview shows the collection, dependent evaluation, and dependent analysis jobs from the tracked submit scripts. Submission is disabled by default and refuses a non-empty run root.

In [ ]:
PREVIEW_PBS_CHAIN = True
SUBMIT_PBS_CHAIN = False
submit_script = {
    'smoke': ROOT / 'cluster' / 'submit_verified_smoke.sh',
    'full': ROOT / 'cluster' / 'submit_verified_full.sh',
}[VERIFIED_MODE]

submit_env = env.copy()
submit_env.update({
    'RUN_NAME': RUN_NAME,
    'RUN_ROOT': str(RUN_ROOT),
    'CLUSTER_LOG_DIR': str(CLUSTER_LOG_DIR),
    'RUN_ID': RUN_NAME.replace('-', '_'),
    'DATASET': DATASET,
    'SWEBENCH_DATASET_REVISION': DATASET_REVISION,
    'SPLIT': SPLIT,
    'EXPECTED_COUNT': str(SMOKE_LIMIT if VERIFIED_MODE == 'smoke' else EXPECTED_COUNT),
    'SMOKE_LIMIT': str(SMOKE_LIMIT),
    'NUM_SHARDS': str(NUM_SHARDS),
    'ROLLOUT_WORKERS': str(ROLLOUT_WORKERS),
    'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
    'MINI_SWE_MODEL': MINI_SWE_MODEL,
    'MINI_SWE_RUNNER': MINI_SWE_RUNNER,
    'MINI_SWE_ENVIRONMENT_CLASS': MINI_SWE_ENVIRONMENT_CLASS,
    'MAX_STEPS': str(FULL_MAX_STEPS),
    'CONTEXT_LENGTH': str(CONTEXT_LENGTH),
    'TEMPERATURE': str(TEMPERATURE),
    'TOP_P': str(TOP_P),
    'TIMEOUT_SECONDS': str(TIMEOUT_SECONDS),
    'OVERWRITE': '0',
})
if TASK_IDS_FILE:
    submit_env['TASK_IDS_FILE'] = TASK_IDS_FILE

if PREVIEW_PBS_CHAIN:
    preview_env = submit_env.copy()
    preview_env['DRY_RUN'] = '1'
    subprocess.run(['bash', str(submit_script)], cwd=ROOT, env=preview_env, check=True)

if SUBMIT_PBS_CHAIN:
    expected_run_root = (scratch / 'runs' / RUN_NAME).resolve()
    if RUN_ROOT != expected_run_root:
        raise ValueError(f'PBS scripts use {expected_run_root}; set RUN_ROOT to that path before submission.')
    if not PREVIEW_PBS_CHAIN:
        raise RuntimeError('Preview the PBS chain before submission.')
    subprocess.run(['bash', str(submit_script)], cwd=ROOT, env=submit_env, check=True)
else:
    print('SUBMIT_PBS_CHAIN is off; no jobs submitted.')

## 4. One-Instance Smoke

This validates the real mini-swe harness, Docker/Apptainer environment, LLM calls, config generation, trajectory capture, and prediction JSONL writing.

In [ ]:
RUN_ONE_INSTANCE_SMOKE = False

def make_rollout_env(output_dir, *, limit='', max_steps=200, num_shards=1, shard_index=0, overwrite=False):
    rollout_env = env.copy()
    # Keep smoke selectors from leaking into full shard runs after notebook re-execution.
    for key in ['LIMIT', 'TASK_IDS_FILE', 'INSTANCE_ID', 'START_INDEX', 'NUM_SHARDS', 'SHARD_INDEX', 'ROLLOUT_WORKERS', 'OVERWRITE']:
        rollout_env.pop(key, None)
    rollout_env.update({
        'DATASET': DATASET,
        'SWEBENCH_DATASET_REVISION': DATASET_REVISION,
        'SPLIT': SPLIT,
        'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
        'HARNESS': 'mini-swe-agent-plus',
        'MINI_SWE_MODEL': MINI_SWE_MODEL,
        'MINI_SWE_RUNNER': MINI_SWE_RUNNER,
        'MINI_SWE_ENVIRONMENT_CLASS': MINI_SWE_ENVIRONMENT_CLASS,
        'MSWEA_SINGULARITY_EXECUTABLE': MSWEA_SINGULARITY_EXECUTABLE,
        'LLM_BASE_URL': LLM_BASE_URL,
        'LLM_API_KEY': LLM_API_KEY,
        'OUTPUT_DIR': str(output_dir),
        'MAX_STEPS': str(max_steps),
        'CONTEXT_LENGTH': str(CONTEXT_LENGTH),
        'TEMPERATURE': str(TEMPERATURE),
        'TOP_P': str(TOP_P),
        'TIMEOUT_SECONDS': str(TIMEOUT_SECONDS),
        'MINI_SWE_WORKERS': str(MINI_SWE_WORKERS),
        'MINI_SWE_DOCKER_START_CONCURRENCY': str(MINI_SWE_DOCKER_START_CONCURRENCY),
        'ROLLOUT_WORKERS': str(ROLLOUT_WORKERS),
        'NUM_SHARDS': str(num_shards),
        'SHARD_INDEX': str(shard_index),
        # Keep mini-swe's nested Rich display out of the notebook; the collector's
        # lightweight tqdm bar still shows overall task completion.
        # Detailed harness output remains available in each trajectory's log file.
        'STREAM_OUTPUT': os.getenv('STREAM_OUTPUT', '0'),
    })
    if limit not in ('', None):
        rollout_env['LIMIT'] = str(limit)
    if TASK_IDS_FILE:
        rollout_env['TASK_IDS_FILE'] = TASK_IDS_FILE
    if MINI_SWE_CONFIG:
        rollout_env['MINI_SWE_CONFIG'] = MINI_SWE_CONFIG
    if overwrite:
        rollout_env['OVERWRITE'] = '1'
    return rollout_env

if RUN_ONE_INSTANCE_SMOKE:
    smoke_env = make_rollout_env(SMOKE_OUTPUT_DIR, limit=1, max_steps=SMOKE_MAX_STEPS, overwrite=False)
    print('OUTPUT_DIR=', smoke_env['OUTPUT_DIR'])
    print('MAX_STEPS=', smoke_env['MAX_STEPS'])
    subprocess.run(['bash', 'scripts/collect_rollouts.sh'], cwd=ROOT, env=smoke_env, check=True)

    summary = json.loads((SMOKE_OUTPUT_DIR / 'summary.json').read_text())
    print(json.dumps({
        'n_tasks': summary['n_tasks'],
        'n_completed': summary['n_completed'],
        'n_errors': summary['n_errors'],
        'n_with_patch': summary['n_with_patch'],
        'results': summary['results'],
    }, indent=2))

## 4.1 Evaluation Helpers

These helpers run the Apptainer SWE-bench evaluator against an existing prediction file. Keep the run switches off until the prediction file exists and you are ready to spend evaluator time.


In [ ]:
EVAL_ROOT = Path(os.getenv('EVAL_ROOT', RUN_ROOT / 'evaluation'))
EVAL_REPORT_DIR = Path(os.getenv('EVAL_REPORT_DIR', EVAL_ROOT / 'reports'))
EVAL_LOG_DIR = Path(os.getenv('EVAL_LOG_DIR', EVAL_ROOT / 'logs'))
EVAL_MAX_WORKERS = int(os.getenv('EVAL_MAX_WORKERS', '20'))
EVAL_TIMEOUT = int(os.getenv('EVAL_TIMEOUT', '3600'))
EVAL_IMAGE_TEMPLATE = os.getenv(
    'SWEBENCH_APPTAINER_IMAGE_TEMPLATE',
    'docker://ghcr.io/epoch-research/swe-bench.eval.x86_64.{instance_id}:latest',
)
EVAL_IMAGE_TEMPLATE = EVAL_IMAGE_TEMPLATE.replace('{instance_id:latest}', '{instance_id}:latest')
EVAL_IMAGE_TEMPLATE = EVAL_IMAGE_TEMPLATE.replace('{instance_id_lower:latest}', '{instance_id_lower}:latest')
EVAL_IMAGE_TEMPLATE = EVAL_IMAGE_TEMPLATE.replace('{image_key:latest}', '{image_key}:latest')


def prediction_instance_ids(predictions_path):
    predictions_path = Path(predictions_path)
    ids = []
    with predictions_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            instance_id = row.get('instance_id')
            if instance_id:
                ids.append(str(instance_id))
    return ids


def write_instance_ids_file(instance_ids, run_id):
    ids_dir = EVAL_ROOT / 'instance-ids'
    ids_dir.mkdir(parents=True, exist_ok=True)
    ids_path = ids_dir / f'{run_id}.txt'
    ids_path.write_text('\n'.join(instance_ids) + ('\n' if instance_ids else ''), encoding='utf-8')
    return ids_path


def make_eval_env(predictions_path, run_id, *, instance_ids=None, max_workers=None, timeout=None, overwrite=False):
    predictions_path = Path(predictions_path)
    if not predictions_path.exists():
        raise FileNotFoundError(predictions_path)

    EVAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)
    EVAL_LOG_DIR.mkdir(parents=True, exist_ok=True)

    eval_env = env.copy()
    eval_env.update({
        'DATASET': DATASET,
        'SWEBENCH_DATASET_REVISION': DATASET_REVISION,
        'SPLIT': SPLIT,
        'AGENTFORGE_MODEL': AGENTFORGE_MODEL,
        'PREDICTIONS_PATH': str(predictions_path),
        'RUN_ID': run_id,
        'REPORT_DIR': str(EVAL_REPORT_DIR),
        'SUMMARY_OUTPUT': str(EVAL_REPORT_DIR / f'{run_id}_summary.json'),
        'LOG_DIR': str(EVAL_LOG_DIR),
        'MAX_WORKERS': str(max_workers or EVAL_MAX_WORKERS),
        'TIMEOUT': str(timeout or EVAL_TIMEOUT),
        'SWEBENCH_APPTAINER_CACHE_DIR': env['SWEBENCH_APPTAINER_CACHE_DIR'],
        'SWEBENCH_APPTAINER_SIF_DIR': env['SWEBENCH_APPTAINER_SIF_DIR'],
        'SWEBENCH_APPTAINER_IMAGE_TEMPLATE': EVAL_IMAGE_TEMPLATE,
    })
    if overwrite:
        eval_env['OVERWRITE'] = '1'
    if instance_ids is not None:
        ids_path = write_instance_ids_file(instance_ids, run_id)
        eval_env['TASK_IDS_FILE'] = str(ids_path)
    return eval_env


def preview_eval(predictions_path, run_id, *, instance_ids=None, max_workers=None):
    eval_env = make_eval_env(
        predictions_path,
        run_id,
        instance_ids=instance_ids,
        max_workers=max_workers,
    )
    preview = {
        'predictions_path': eval_env['PREDICTIONS_PATH'],
        'run_id': eval_env['RUN_ID'],
        'summary_output': eval_env['SUMMARY_OUTPUT'],
        'report_dir': eval_env['REPORT_DIR'],
        'log_dir': eval_env['LOG_DIR'],
        'task_ids_file': eval_env.get('TASK_IDS_FILE'),
        'max_workers': eval_env['MAX_WORKERS'],
        'timeout': eval_env['TIMEOUT'],
        'image_template': eval_env['SWEBENCH_APPTAINER_IMAGE_TEMPLATE'],
    }
    print(json.dumps(preview, indent=2))
    return preview


def run_apptainer_eval(predictions_path, run_id, *, instance_ids=None, max_workers=None, timeout=None, overwrite=False):
    eval_env = make_eval_env(
        predictions_path,
        run_id,
        instance_ids=instance_ids,
        max_workers=max_workers,
        timeout=timeout,
        overwrite=overwrite,
    )
    print('Evaluating:', eval_env['PREDICTIONS_PATH'])
    print('Run id:', run_id)
    if eval_env.get('TASK_IDS_FILE'):
        print('Restricting to ids in:', eval_env['TASK_IDS_FILE'])
    subprocess.run(['bash', 'scripts/evaluate_apptainer.sh'], cwd=ROOT, env=eval_env, check=True)
    summary_path = Path(eval_env['SUMMARY_OUTPUT'])
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))
    return summary


## 4.2 Evaluate Smoke Prediction

Run this immediately after the one-instance smoke has produced `SMOKE_OUTPUT_DIR / 'predictions.jsonl'`. It evaluates only the submitted smoke instance, so this is the cheapest way to prove the evaluator path before launching a full shard.


In [ ]:
RUN_SMOKE_EVAL = False
SMOKE_EVAL_RUN_ID = os.getenv('SMOKE_EVAL_RUN_ID', f'{RUN_NAME.replace("-", "_")}_smoke')
smoke_predictions = SMOKE_OUTPUT_DIR / 'predictions.jsonl'

if smoke_predictions.exists():
    smoke_ids = prediction_instance_ids(smoke_predictions)
    print(f'Smoke predictions: {smoke_predictions}')
    print(f'Smoke submitted ids: {smoke_ids}')
    if RUN_SMOKE_EVAL:
        smoke_eval_summary = run_apptainer_eval(
            smoke_predictions,
            SMOKE_EVAL_RUN_ID,
            instance_ids=smoke_ids,
            max_workers=1,
        )
    else:
        preview_eval(smoke_predictions, SMOKE_EVAL_RUN_ID, instance_ids=smoke_ids, max_workers=1)
        print('Dry preview only. Set RUN_SMOKE_EVAL = True to run the evaluator.')
else:
    print(f'No smoke predictions yet: {smoke_predictions}')


## 5. Run One Full Shard

Keep `RUN_FULL_SHARD = False` until the smoke run behaves sensibly. A shard uses `index % NUM_SHARDS == SHARD_INDEX`, so independent jobs can run safely into separate `shard-*` folders.

In [ ]:
RUN_FULL_SHARD = False
FULL_LIMIT = ''  # Empty means all tasks assigned to this shard. Set manually, e.g. '20', for a small test.
ROLLOUT_WORKERS = int(globals().get('ROLLOUT_WORKERS', os.getenv('ROLLOUT_WORKERS', str(MINI_SWE_WORKERS))))
shard_output = SHARD_ROOT / f'shard-{SHARD_INDEX}'

full_env = make_rollout_env(
    shard_output,
    limit=FULL_LIMIT,
    max_steps=FULL_MAX_STEPS,
    num_shards=NUM_SHARDS,
    shard_index=SHARD_INDEX,
    overwrite=os.getenv('OVERWRITE', '0') == '1',
)

# This cell should be safe to rerun even if the current kernel still has an older make_rollout_env().
for key in ['LIMIT', 'TASK_IDS_FILE', 'INSTANCE_ID', 'START_INDEX']:
    full_env.pop(key, None)
full_env['ROLLOUT_WORKERS'] = str(ROLLOUT_WORKERS)
if FULL_LIMIT not in ('', None):
    full_env['LIMIT'] = str(FULL_LIMIT)

selection_filters = {key: full_env.get(key) for key in ['LIMIT', 'TASK_IDS_FILE', 'INSTANCE_ID', 'START_INDEX'] if full_env.get(key)}

print(json.dumps({
    'RUN_FULL_SHARD': RUN_FULL_SHARD,
    'OUTPUT_DIR': full_env['OUTPUT_DIR'],
    'NUM_SHARDS': full_env['NUM_SHARDS'],
    'SHARD_INDEX': full_env['SHARD_INDEX'],
    'MAX_STEPS': full_env['MAX_STEPS'],
    'MINI_SWE_WORKERS': full_env['MINI_SWE_WORKERS'],
    'MINI_SWE_DOCKER_START_CONCURRENCY': full_env['MINI_SWE_DOCKER_START_CONCURRENCY'],
    'ROLLOUT_WORKERS': full_env['ROLLOUT_WORKERS'],
    'selection_filters': selection_filters or '<none>',
    'FULL_LIMIT': FULL_LIMIT or '<all tasks in shard>',
}, indent=2))

if RUN_FULL_SHARD:
    subprocess.run(['bash', 'scripts/collect_rollouts.sh'], cwd=ROOT, env=full_env, check=True)
    print((shard_output / 'summary.json').read_text())
else:
    print('Dry preview only. Set RUN_FULL_SHARD = True to launch this shard from the notebook.')

## 5.1 Evaluate Current Shard

Use this after section 5 finishes a shard and writes `shard_output / 'predictions.jsonl'`. It restricts evaluation to the submitted ids in that shard so partial shard checks do not report hundreds of unrelated missing instances.


In [ ]:
RUN_SHARD_EVAL = False
SHARD_EVAL_RUN_ID = os.getenv('SHARD_EVAL_RUN_ID', f'{RUN_NAME.replace("-", "_")}_shard_{SHARD_INDEX}')
shard_predictions = shard_output / 'predictions.jsonl'

if shard_predictions.exists():
    shard_ids = prediction_instance_ids(shard_predictions)
    print(f'Shard predictions: {shard_predictions}')
    print(f'Shard submitted count: {len(shard_ids)}')
    if RUN_SHARD_EVAL:
        shard_eval_summary = run_apptainer_eval(
            shard_predictions,
            SHARD_EVAL_RUN_ID,
            instance_ids=shard_ids,
            max_workers=EVAL_MAX_WORKERS,
        )
    else:
        preview_eval(shard_predictions, SHARD_EVAL_RUN_ID, instance_ids=shard_ids)
        print('Dry preview only. Set RUN_SHARD_EVAL = True to evaluate this shard.')
else:
    print(f'No shard predictions yet: {shard_predictions}')


## 6. Merge Shards

Run after some or all shard folders contain `predictions.jsonl`. For official scoring, feed the merged file into the evaluation job.

In [ ]:
prediction_paths = sorted(glob.glob(str(SHARD_ROOT / 'shard-*' / 'predictions.jsonl')))
print('\n'.join(prediction_paths) if prediction_paths else 'No shard prediction files found yet.')

RUN_MERGE = False
if RUN_MERGE and prediction_paths:
    merged_dir = MERGED_DIR
    merged_dir.mkdir(parents=True, exist_ok=True)
    merge_env = env.copy()
    merge_env.update({
        'OUTPUT': str(merged_dir / 'predictions.jsonl'),
        'SUMMARY_OUTPUT': str(merged_dir / 'predictions_summary.json'),
    })
    subprocess.run(['bash', 'scripts/merge_predictions.sh', *prediction_paths], cwd=ROOT, env=merge_env, check=True)
    print((merged_dir / 'predictions_summary.json').read_text())
elif RUN_MERGE:
    raise RuntimeError('No shard prediction files found to merge.')

## 6.1 Evaluate Merged Predictions

Run this after section 6 merges shard prediction files. For partial progress, keep `MERGED_SUBMITTED_ONLY = True`; for the final official-style 500-instance report, set it to `False` once all shards are present.


In [ ]:
RUN_MERGED_EVAL = False
MERGED_EVAL_RUN_ID = os.getenv('MERGED_EVAL_RUN_ID', RUN_NAME.replace('-', '_'))
MERGED_SUBMITTED_ONLY = True
merged_predictions = MERGED_DIR / 'predictions.jsonl'

if merged_predictions.exists():
    merged_ids = prediction_instance_ids(merged_predictions)
    eval_ids = merged_ids if MERGED_SUBMITTED_ONLY else None
    print(f'Merged predictions: {merged_predictions}')
    print(f'Merged submitted count: {len(merged_ids)}')
    print(f'MERGED_SUBMITTED_ONLY: {MERGED_SUBMITTED_ONLY}')
    if RUN_MERGED_EVAL:
        merged_eval_summary = run_apptainer_eval(
            merged_predictions,
            MERGED_EVAL_RUN_ID,
            instance_ids=eval_ids,
            max_workers=EVAL_MAX_WORKERS,
        )
    else:
        preview_eval(merged_predictions, MERGED_EVAL_RUN_ID, instance_ids=eval_ids)
        print('Dry preview only. Set RUN_MERGED_EVAL = True to evaluate merged predictions.')
else:
    print(f'No merged predictions yet: {merged_predictions}')


## Runtime Sizing

- Upper-bound model calls: `n_instances * MAX_STEPS`. For SWE-bench Verified, `500 * 200 = 100,000` chat completions.
- Wall time is approximately `ceil(n_instances / effective_parallel_instances) * average_instance_time`.
- `NUM_SHARDS` parallelizes across jobs/nodes. This is the cleanest scaling path.
- `ROLLOUT_WORKERS` parallelizes multiple selected SWE-bench instances inside one shard process. Use it only when the LLM server, GPU memory, and Apptainer startup can handle concurrent instances.
- `MINI_SWE_WORKERS` is still passed to the mini-swe command, but this wrapper invokes mini-swe with one filtered instance at a time, so `ROLLOUT_WORKERS` is the shard-concurrency knob that matters here.
- Evaluation is separate and also parallelizable with `MAX_WORKERS`, but rollout generation usually dominates when using 200 steps.

After the one-instance smoke, use its elapsed time as your best site-specific estimate. If one 200-step instance takes 20 minutes and you run 50 effective parallel instances, 500 tasks take about `(500 / 50) * 20 = 200 minutes`, plus model startup, image pulls, queue overhead, and stragglers. Compatible reruns resume completed slots; change `RUN_NAME` when result-affecting settings change.